# 10｜密度分群與機率模型

從上到下執行程式格。第一格安裝缺少的套件，第二格準備本章所需設定。

## 投影片與程式的閱讀方式

對應 `slides/10_密度分群與機率模型.md`，目前 44 頁。逐圖完整表見 `programs/SLIDE_NOTEBOOK_MAP_05_25.md`。

DBSCAN 延伸、共變異形狀、穩定 EM、似然/MAP、階層樹與異常評估切分。

保留教材原圖；同概念重畫不保證原書分數相同。架構圖以機制或形狀對照，完整模型案例的範圍在各格說明。

| 程式代碼 | 投影片頁次 | 本格學習內容 |
|---|---|---|
| 10-03 | 4、5、6、7 | 1. DBSCAN：核心點把密集區連成一群 |
| 10-04 | 8 | 2. 手算驗算：誰是核心，誰是雜訊？ |
| 10-05 | 9、10 | 3. DBSCAN 沒有原生 predict：延伸要另訂規則 |
| 10-06 | 3、16、17、18、23 | 4. GMM 與 EM：先選成分，再從該高斯取樣 |
| 10-07 | 19、20、21、22 | 5. 手算驗算：軟指派與一次更新 |
| 10-08 | 25、26、35 | 6. 限制共變異形狀：full、tied、spherical |
| 10-09 | 29、30、31、34、36、37 | 7. 密度異常與 AIC／BIC |
| 10-10 | 41、42、43 | 8. 接回 PCA：用重建誤差找異常 |
| 10-11 | 9、10、25、26、34、35、37、38、39、44 | 9. 延伸分區、共變異形狀、AIC／BIC 與彎月 |
| 10-12 | 16、17、19、20、21、22、23、24、28、32、33、44 | 10. 手刻穩定 EM、密度與似然 |
| 10-13 | 11、12、13、14、30、31、40、41、42、44 | 11. 階層樹、圖分群與新穎偵測 |

In [ ]:
# 第 1 格：安裝本 Notebook 實際使用的套件。
# 程式 10-01｜投影片 10：2 頁
import importlib.util, subprocess, sys
_needed = {'scipy': 'scipy>=1.11,<2', 'numpy': 'numpy>=1.26,<3', 'pandas': 'pandas>=2.0,<3', 'matplotlib': 'matplotlib>=3.7,<4', 'sklearn': 'scikit-learn>=1.4,<2'}
_missing = [spec for module, spec in _needed.items() if importlib.util.find_spec(module) is None]
if _missing:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *_missing])

In [ ]:
# 第 2 格：本 Notebook 的輸出位置與繪圖設定。
# 程式 10-02｜投影片 10：2 頁
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
ROOT = Path.cwd()
if ROOT.name == "notebooks" and (ROOT.parent / "pyproject.toml").exists():
    ROOT = ROOT.parent
OUT = ROOT / ("outputs" if (ROOT / "pyproject.toml").exists() else "mlcourse_outputs")
plt.rcParams.update({"figure.figsize": (8, 4.5), "figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.2, "font.size": 11})

def savefig(name):
    folder = OUT / "figures"
    folder.mkdir(parents=True, exist_ok=True)
    plt.gcf().savefig(folder / f"{name}.png", dpi=150, bbox_inches="tight")
    svg_path = folder / f"{name}.svg"
    plt.gcf().savefig(svg_path, bbox_inches="tight", metadata={"Date": None})
    svg_path.write_text("\n".join(line.rstrip() for line in svg_path.read_text().splitlines()) + "\n")
    plt.show()
    plt.close(plt.gcf())

def report(name, values):
    folder = OUT / "reports"
    folder.mkdir(parents=True, exist_ok=True)
    (folder / f"{name}.json").write_text(
        json.dumps(values, ensure_ascii=False, indent=2, default=float) + "\n")

## 1. DBSCAN：核心點把密集區連成一群

`labels_` 中的 `-1` 是雜訊；`core_sample_indices_` 是核心點。改 `eps` 會讓同一資料從破碎變連續。

In [ ]:
# 程式 10-03｜投影片 10：4、5、6、7 頁
from sklearn.datasets import make_moons
from sklearn.cluster import DBSCAN
Xm, _ = make_moons(n_samples=500, noise=.10, random_state=SEED)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
rows = []
for ax, eps in zip(axes, [0.08, 0.15, 0.30]):
    db = DBSCAN(eps=eps, min_samples=5).fit(Xm)
    n_clusters = len(set(db.labels_[db.labels_ != -1]))
    ax.scatter(Xm[:, 0], Xm[:, 1], c=db.labels_, cmap='tab10', s=10)
    ax.set_title(f'eps={eps}: {n_clusters} clusters, {np.sum(db.labels_ == -1)} noise')
    rows.append({'eps': eps, 'clusters': n_clusters, 'noise': int(np.sum(db.labels_ == -1)),
                 'core_points': len(db.core_sample_indices_)})
savefig('15_dbscan_eps')
pd.DataFrame(rows).set_index('eps')

## 2. 手算驗算：誰是核心，誰是雜訊？

對應投影片活動。一維點 `[0, .1, .2, .4, 2.0]`，`eps=.21`、`min_samples=3`（鄰域含自己）。

In [ ]:
# 程式 10-04｜投影片 10：8 頁
pts = np.array([0., .1, .2, .4, 2.0]); eps = .21
neigh = (np.abs(pts[:, None] - pts[None]) <= eps).sum(axis=1)
assert neigh.tolist() == [3, 3, 4, 2, 1]
def classify(min_samples):
    core = neigh >= min_samples
    kind = []
    for i in range(len(pts)):
        if core[i]:
            kind.append('core')
        elif np.any(core & (np.abs(pts - pts[i]) <= eps)):
            kind.append('border')
        else:
            kind.append('noise')
    return kind
k3, k4 = classify(3), classify(4)
assert k3 == ['core', 'core', 'core', 'border', 'noise']
assert k4 == ['border', 'border', 'core', 'border', 'noise']
pd.DataFrame({'point': pts, 'neighbors': neigh, 'min_samples=3': k3, 'min_samples=4': k4})

## 3. DBSCAN 沒有原生 predict：延伸要另訂規則

一種延伸是用核心點的最近鄰投票，並拒絕距離所有核心點都超過 `eps` 的新點。

In [ ]:
# 程式 10-05｜投影片 10：9、10 頁
from sklearn.neighbors import KNeighborsClassifier
db = DBSCAN(eps=0.18, min_samples=5).fit(Xm)
core_mask = np.zeros(len(Xm), bool); core_mask[db.core_sample_indices_] = True
knn = KNeighborsClassifier(n_neighbors=5).fit(Xm[core_mask], db.labels_[core_mask])
from sklearn.neighbors import NearestNeighbors
nn = NearestNeighbors(n_neighbors=1).fit(Xm[core_mask])
new_pts = np.array([[0.0, 0.3], [1.0, -0.3], [2.5, 2.5]])   # 最後一點刻意很遠
dist, _ = nn.kneighbors(new_pts)
pred = knn.predict(new_pts)
pred = np.where(dist[:, 0] <= 0.18, pred, -1)               # 太遠 -> 拒絕
assert pred[-1] == -1 and np.all(pred[:2] != -1)
pd.DataFrame({'x': new_pts[:, 0], 'y': new_pts[:, 1], 'dist_to_core': dist[:, 0].round(3), 'assigned': pred})

## 4. GMM 與 EM：先選成分，再從該高斯取樣

`GaussianMixture` 同時給群邊界與密度等高線。`converged_`、`n_iter_` 顯示 EM 是否收斂。

In [ ]:
# 程式 10-06｜投影片 10：3、16、17、18、23 頁
from sklearn.mixture import GaussianMixture
from matplotlib.patches import Ellipse
rng = np.random.default_rng(SEED)
G = np.vstack([rng.multivariate_normal([0, 0], [[1, .8], [.8, 1]], 250),
               rng.multivariate_normal([4, 5], [[1, -.6], [-.6, 1]], 250),
               rng.multivariate_normal([-3, 4], [[.5, 0], [0, 2]], 200)])
gm = GaussianMixture(n_components=3, n_init=10, covariance_type='full', random_state=SEED).fit(G)
xx, yy = np.meshgrid(np.linspace(G[:, 0].min() - 1, G[:, 0].max() + 1, 200),
                     np.linspace(G[:, 1].min() - 1, G[:, 1].max() + 1, 200))
dens = np.exp(gm.score_samples(np.c_[xx.ravel(), yy.ravel()])).reshape(xx.shape)
plt.contour(xx, yy, dens, levels=8, cmap='Greys')
plt.scatter(G[:, 0], G[:, 1], c=gm.predict(G), cmap='Dark2', s=10)
for k in range(3):
    v, w = np.linalg.eigh(gm.covariances_[k])
    ang = np.degrees(np.arctan2(w[1, 0], w[0, 0]))
    plt.gca().add_patch(Ellipse(gm.means_[k], 2 * 2 * np.sqrt(v[0]), 2 * 2 * np.sqrt(v[1]),
                                angle=ang, fill=False, edgecolor='red'))
plt.title(f'GMM: converged={gm.converged_}, iters={gm.n_iter_}')
savefig('15_gmm_ellipses')
print('weights:', np.round(gm.weights_, 3))

## 5. 手算驗算：軟指派與一次更新

對應投影片活動。E-step：兩成分權重皆 `.5`，某點在兩成分下的密度為 `.2` 與 `.05`。M-step：資料 `[0,2]`，成分 A 的責任值 `[.8, .2]`。

In [ ]:
# 程式 10-07｜投影片 10：19、20、21、22 頁
w = np.array([.5, .5]); dens_pt = np.array([.2, .05])
resp = w * dens_pt / np.sum(w * dens_pt)
assert np.allclose(resp, [.8, .2])

data = np.array([0., 2.]); rA = np.array([.8, .2])
NA = rA.sum(); piA = NA / len(data)
muA = np.sum(rA * data) / NA
varA = np.sum(rA * (data - muA) ** 2) / NA
assert np.isclose(NA, 1.0) and np.isclose(piA, .5) and np.isclose(muA, .4) and np.isclose(varA, .64)
pd.Series({'E_step_responsibilities': resp.tolist(), 'N_A': NA, 'new_weight_A': piA,
           'new_mean_A': muA, 'new_variance_A': varA})

## 6. 限制共變異形狀：full、tied、spherical

限制形狀可降低參數量與過擬合風險，但也可能切出不符合資料的群。

In [ ]:
# 程式 10-08｜投影片 10：25、26、35 頁
rows = []
for ct in ['full', 'tied', 'spherical', 'diag']:
    g = GaussianMixture(n_components=3, n_init=5, covariance_type=ct, random_state=SEED).fit(G)
    rows.append({'covariance_type': ct, 'n_parameters': (3 - 1) + 3 * 2 + {'full': 3 * 3, 'tied': 3, 'spherical': 3, 'diag': 3 * 2}[ct],
                 'mean_log_likelihood': g.score(G), 'BIC': g.bic(G)})
cov_table = pd.DataFrame(rows).set_index('covariance_type')
assert cov_table.loc['spherical', 'n_parameters'] < cov_table.loc['full', 'n_parameters']
cov_table

## 7. 密度異常與 AIC／BIC

`score_samples` 給每點的對數密度；取低百分位當閾值標出異常。AIC／BIC 對複雜度的懲罰不同，可能給不同答案。

In [ ]:
# 程式 10-09｜投影片 10：29、30、31、34、36、37 頁
# 投影片數例：A p=5, loglik=-100；B p=8, loglik=-96；m=100
m = 100
aic = lambda p, ll: 2 * p - 2 * ll
bic = lambda p, ll: p * np.log(m) - 2 * ll
assert aic(5, -100) == 210 and aic(8, -96) == 208
assert np.isclose(bic(5, -100), 223.02585) and np.isclose(bic(8, -96), 228.84136)
print('AIC prefers B (208 < 210); BIC prefers A (223.0 < 228.8)')

# 合成資料：用擬合資料示範尾端 2% 的定義，不能當獨立偵測成效
log_density = gm.score_samples(G)
threshold = np.percentile(log_density, 2)
flagged = log_density < threshold
plt.scatter(G[:, 0], G[:, 1], c='lightgray', s=10)
plt.scatter(G[flagged, 0], G[flagged, 1], c='red', s=30, label='low-density (bottom 2%)')
plt.legend(); plt.title('GMM density anomalies')
savefig('15_gmm_anomaly')

# 選成分數：AIC / BIC 曲線
ns = range(1, 8)
aics = [GaussianMixture(n, n_init=5, random_state=SEED).fit(G).aic(G) for n in ns]
bics = [GaussianMixture(n, n_init=5, random_state=SEED).fit(G).bic(G) for n in ns]
report('density_probabilistic', {'neighbor_counts': neigh.tolist(),
       'em_responsibilities': resp.tolist(), 'em_update_A': {'mean': muA, 'variance': varA},
       'aic_bic_worked_example': {'A': [aic(5, -100), bic(5, -100)], 'B': [aic(8, -96), bic(8, -96)]},
       'covariance_types': cov_table.to_dict(),
       'best_n_by_bic': int(list(ns)[int(np.argmin(bics))]), 'anomaly_threshold': float(threshold)})
pd.DataFrame({'n_components': list(ns), 'AIC': np.round(aics, 1), 'BIC': np.round(bics, 1)}).set_index('n_components')

## 8. 接回 PCA：用重建誤差找異常

在正常資料學到的低維子空間裡難以重建的點，重建誤差較高。這是另一種分數，不是 GMM 密度。

這裡的正常資料保留集用於校準閾值，回報的比例不是獨立測試誤報率。正式評估還須另留正常／異常測試集。

In [ ]:
# 程式 10-10｜投影片 10：41、42、43 頁
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.decomposition import PCA
digits = load_digits()
X8 = digits.data[digits.target == 8]                     # 只用數字 8 當「正常」
Xtr, Xval = train_test_split(X8, test_size=.3, random_state=SEED)
pca = PCA(n_components=12, random_state=SEED).fit(Xtr)
def recon_error(X):
    return np.mean((X - pca.inverse_transform(pca.transform(X))) ** 2, axis=1)
thr = np.percentile(recon_error(Xval), 95)
others = digits.data[digits.target != 8][:300]           # 非 8 應多半被標出
flag_rate_normal = float(np.mean(recon_error(Xval) > thr))
flag_rate_other = float(np.mean(recon_error(others) > thr))
assert flag_rate_other > flag_rate_normal
print(f'flag rate: threshold-calibration 8s = {flag_rate_normal:.2f}, non-8 digits = {flag_rate_other:.2f}')

## 9. 延伸分區、共變異形狀、AIC／BIC 與彎月

新點分類先以核心點投票，再按距離拒絕；GMM 對彎月的成分切割不等於找到兩個語意群。Bayesian GMM 的成分數是上限，權重門檻只是讀圖工具。

In [ ]:
# 程式 10-11｜投影片 10：9、10、25、26、34、35、37、38、39、44 頁
grid10 = np.c_[xx.ravel(), yy.ravel()]
# 延伸 DBSCAN 的格網另以彎月範圍建立
u10, v10 = np.meshgrid(np.linspace(-1.5,2.5,160), np.linspace(-1,1.5,130)); q10=np.c_[u10.ravel(),v10.ravel()]
d10,_=nn.kneighbors(q10); z10=np.where(d10[:,0]<=.18, knn.predict(q10), -1)
plt.contourf(u10,v10,z10.reshape(u10.shape),alpha=.3,cmap='tab10'); plt.scatter(*Xm.T,c=db.labels_,s=8,cmap='tab10')
plt.title('Core-point extension with distance rejection'); savefig('c10_dbscan_extension')
fig, axes=plt.subplots(1,3,figsize=(12,3.6))
for ax, ct in zip(axes,['full','tied','spherical']):
    g10=GaussianMixture(3,covariance_type=ct,n_init=5,random_state=SEED).fit(G)
    ax.contour(xx,yy,np.exp(g10.score_samples(grid10)).reshape(xx.shape),levels=8)
    ax.scatter(*G.T,c=g10.predict(G),s=6,cmap='Dark2');ax.set_title(ct)
savefig('c10_covariance_shapes')
fig, axes=plt.subplots(1,2,figsize=(11,4))
for ct in ['full','tied','diag','spherical']:
    fitted10=[GaussianMixture(k,covariance_type=ct,n_init=3,random_state=SEED).fit(G) for k in ns]
    axes[0].plot(list(ns),[m.aic(G) for m in fitted10],label=ct)
    axes[1].plot(list(ns),[m.bic(G) for m in fitted10],label=ct)
for ax,title in zip(axes,['AIC','BIC']):ax.set(xlabel='components',ylabel=title);ax.legend()
savefig('c10_information_criteria')
from sklearn.mixture import BayesianGaussianMixture
fig,axes=plt.subplots(1,2,figsize=(10,4))
for ax,g10 in zip(axes,[GaussianMixture(8,n_init=3,random_state=SEED),BayesianGaussianMixture(n_components=8,max_iter=500,random_state=SEED)]):
    g10.fit(Xm);ax.scatter(*Xm.T,c=g10.predict(Xm),s=8,cmap='tab10');ax.set_title(type(g10).__name__)
    print(type(g10).__name__,'weights:',g10.weights_.round(3))
savefig('c10_moons_components')

## 10. 手刻穩定 EM、密度與似然

一維兩成分 EM 用 log-sum-exp 算責任值；變異下限避免退化。似然固定資料、改參數；密度固定參數、改位置。MAP 另外加入先驗，不等於最大似然。

In [ ]:
# 程式 10-12｜投影片 10：16、17、19、20、21、22、23、24、28、32、33、44 頁
from scipy.special import logsumexp
from scipy.stats import norm
rng10=np.random.default_rng(SEED); data10=np.r_[rng10.normal(-2,.6,100),rng10.normal(2,1,150)]
mu10=np.array([-1.,1.]);var10=np.ones(2);mix10=np.ones(2)/2;trace10=[]
for iteration10 in range(50):
    logjoint10=np.log(mix10)+norm.logpdf(data10[:,None],mu10,np.sqrt(var10))
    logden10=logsumexp(logjoint10,axis=1);trace10.append(logden10.sum())
    r10=np.exp(logjoint10-logden10[:,None]);Nk10=r10.sum(0)
    mu10=(r10*data10[:,None]).sum(0)/Nk10
    var10=np.maximum((r10*(data10[:,None]-mu10)**2).sum(0)/Nk10,1e-6);mix10=Nk10/len(data10)
assert np.allclose(r10.sum(1),1) and np.min(np.diff(trace10))>-1e-7
plt.plot(trace10);plt.xlabel('EM iteration');plt.ylabel('log likelihood');savefig('c10_em_convergence')
observed10=np.array([1.,2.,3.]);candidates10=np.linspace(-1,4,250)
likelihood10=norm.logpdf(observed10[:,None],loc=candidates10).sum(0)
posterior10=likelihood10+norm.logpdf(candidates10,loc=0,scale=.5)
fig,axes=plt.subplots(1,2,figsize=(10,4))
axes[0].plot(candidates10,norm.pdf(candidates10,loc=2));axes[0].set(xlabel='possible observation x',ylabel='density; mean fixed at 2')
axes[1].plot(candidates10,likelihood10,label='log likelihood');axes[1].plot(candidates10,posterior10,label='plus log prior');axes[1].set(xlabel='candidate mean',ylabel='score on fixed [1,2,3]');axes[1].legend()
savefig('c10_density_vs_likelihood')
print('GMM posterior / log density:',gm.predict_proba(G[:2]),gm.score_samples(G[:2]))
samples10,component10=gm.sample(4);print('Generated samples and component IDs:',samples10,component10)

## 11. 階層樹、圖分群與新穎偵測

用相同小型資料比較連結方式與密度方法。新穎偵測的 fit、閾值校準、測試分開；此處只有合成異常，不能推論真實部署偵測率。

In [ ]:
# 程式 10-13｜投影片 10：11、12、13、14、30、31、40、41、42、44 頁
from scipy.cluster.hierarchy import linkage,dendrogram
from sklearn.cluster import MeanShift,SpectralClustering,AgglomerativeClustering
small10=G[::20];fig,axes=plt.subplots(1,3,figsize=(13,4))
for ax,method in zip(axes,['single','complete','ward']):
    dendrogram(linkage(small10,method=method),ax=ax,no_labels=True);ax.set(title=method,ylabel='merge distance')
savefig('c10_linkage_trees')
print('MeanShift cluster count:',len(np.unique(MeanShift().fit_predict(small10))))
print('Spectral cluster sizes:',np.bincount(SpectralClustering(3,affinity='nearest_neighbors',n_neighbors=8,random_state=SEED).fit_predict(small10)))
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
normal10=rng10.normal(size=(300,2)); calibration10=rng10.normal(size=(100,2))
test_normal10=rng10.normal(size=(100,2));test_anomaly10=rng10.normal(6,1,size=(50,2));test10=np.vstack([test_normal10,test_anomaly10])
for detector10 in [IsolationForest(random_state=SEED),LocalOutlierFactor(novelty=True),OneClassSVM(gamma='scale')]:
    detector10.fit(normal10);cut10=np.percentile(detector10.score_samples(calibration10),5)
    flag10=detector10.score_samples(test10)<cut10
    print(type(detector10).__name__,'test FPR/TPR:',flag10[:100].mean(),flag10[100:].mean())

## 練習與參考答案

- 可以用非核心的邊界點把 DBSCAN 延伸到新點嗎？**此處採核心點規則；其他外加分類器可另訂規則，但不能說是 DBSCAN 原生 predict。**
- 責任值與密度一樣嗎？**否，責任值是成分後驗、每列和為 1；密度是單位體積的量。**
- AIC 與 BIC 給不同答案時怎麼辦？**它們是相對選擇準則，不是正確率；BIC 對參數增加懲罰較重，需結合用途判斷。**
- 重建誤差一定能找到所有異常嗎？**否，若異常剛好落在保留的子空間內就偵測不到。**